# Добавляю geoname_id, population, координаты

In [2]:
import pandas as pd
import geonamescache
import country_converter as coco

In [3]:
df_2026_final = pd.read_pickle("./data/matched_df_2026.pkl")

In [4]:
df_2026_final

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,local_purchasing_power,quality_of_life,property_price_to_income_ratio,traffic_commute_time,climate,safety,health_care,pollution,link
city_id,,,,,,,,,,,,,,,,,,,
3,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,176.4,204.6,11.9,36.9,81.5,76.7,70.1,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...
8,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,161.3,200.5,13.0,32.6,82.6,70.5,69.9,24.2,https://www.numbeo.com/cost-of-living/in/Genev...
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,186.9,206.3,9.8,36.9,82.8,68.9,69.3,31.4,https://www.numbeo.com/cost-of-living/in/Basel...
535,Lugano,NaN,NaN,CHE,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,160.3,NaN,NaN,NaN,NaN,78.4,NaN,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...
5,Lausanne,NaN,NaN,CHE,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,181.2,206.4,11.0,33.3,73.3,71.6,70.3,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,Indore,NaN,NaN,IND,India,"Indore, India",18.1,4.0,11.6,20.5,68.7,NaN,NaN,NaN,NaN,51.8,NaN,60.2,https://www.numbeo.com/cost-of-living/in/Indor...
621,Guwahati,NaN,NaN,IND,India,"Guwahati, India",17.7,3.3,11.0,21.3,62.9,NaN,NaN,NaN,NaN,NaN,NaN,75.8,https://www.numbeo.com/cost-of-living/in/Guwah...
622,Lucknow,NaN,NaN,IND,India,"Lucknow (Lakhnau), India",17.5,3.4,11.0,19.8,69.8,NaN,NaN,NaN,NaN,54.5,62.9,77.9,https://www.numbeo.com/cost-of-living/in/Luckn...


In [5]:
df_2026_final_full = df_2026_final.copy()

In [6]:
converter = coco.CountryConverter()

df_2026_final_full["country_code2"] = converter.convert(
    names=df_2026_final_full["country_code"], src="ISO3", to="ISO2"
)

gc = geonamescache.GeonamesCache()


def pick_best(candidates):
    if not candidates:
        return None
    return max(candidates, key=lambda x: int(x.get("population", 0) or 0))


def extract_result(city=None, match_count=0, match_source=None):
    if city is None:
        return pd.Series(
            {
                "geonameid": None,
                "population": None,
                "latitude": None,
                "longitude": None,
                "timezone": None,
                "match_count": match_count,
                "match_source": match_source,
            }
        )

    return pd.Series(
        {
            "geonameid": int(city["geonameid"])
            if city.get("geonameid") is not None
            else None,
            "population": int(city["population"])
            if city.get("population") not in (None, "", 0, "0")
            else None,
            "latitude": float(city["latitude"])
            if city.get("latitude") not in (None, "")
            else None,
            "longitude": float(city["longitude"])
            if city.get("longitude") not in (None, "")
            else None,
            "timezone": city.get("timezone"),
            "match_count": match_count,
            "match_source": match_source,
        }
    )


def city_variants(city):
    variants = []
    if pd.isna(city):
        return variants

    city = str(city).strip()
    if not city:
        return variants

    variants.append(city)

    for sep in ["-", "/", ","]:
        if sep in city:
            left = city.split(sep)[0].strip()
            if left:
                variants.append(left)

    return list(dict.fromkeys(variants))


def filter_candidates(matches, country_code2, state_code):
    filtered = [c for c in matches if c.get("countrycode") == country_code2]

    if not filtered:
        return []

    if country_code2 == "US" and pd.notna(state_code):
        state_filtered = [c for c in filtered if c.get("admin1code") == state_code]
        if state_filtered:
            return state_filtered

    return filtered


def search_exact_name(city, country_code2, state_code):
    matches = gc.search_cities(
        city,
        attribute="name",
        case_sensitive=False,
        contains_search=False,
    )
    return filter_candidates(matches, country_code2, state_code)


def search_exact_alternatenames(city, country_code2, state_code):
    matches = gc.search_cities(
        city,
        attribute="alternatenames",
        case_sensitive=False,
        contains_search=False,
    )
    return filter_candidates(matches, country_code2, state_code)


def match_city(row):
    city = row["city"]
    country_code2 = row["country_code2"]
    state_code = row["state_code"]

    if pd.isna(city) or pd.isna(country_code2):
        return extract_result(None, match_count=0, match_source="invalid_input")

    variants = city_variants(city)

    for variant in variants:
        matches = search_exact_name(variant, country_code2, state_code)
        if matches:
            best = pick_best(matches)
            return extract_result(
                best, match_count=len(matches), match_source=f"name_exact:{variant}"
            )

    for variant in variants:
        matches = search_exact_alternatenames(variant, country_code2, state_code)
        if matches:
            best = pick_best(matches)
            return extract_result(
                best, match_count=len(matches), match_source=f"alt_exact:{variant}"
            )

    return extract_result(None, match_count=0, match_source="not_found")


geo_data = df_2026_final_full.apply(match_city, axis=1)

df_2026_final_full = pd.concat([df_2026_final_full, geo_data], axis=1)


In [7]:
df_2026_final_full

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,...,pollution,link,country_code2,geonameid,population,latitude,longitude,timezone,match_count,match_source
city_id,,,,,,,,,,,,,,,,,,,,,
3,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,...,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CH,2657896.0,415367.0,47.36667,8.55000,Europe/Zurich,1,alt_exact:Zurich
8,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,...,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CH,2660646.0,201741.0,46.20222,6.14569,Europe/Zurich,1,alt_exact:Geneva
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,...,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CH,2661604.0,177595.0,47.55839,7.57327,Europe/Zurich,1,name_exact:Basel
535,Lugano,NaN,NaN,CHE,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,...,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...,CH,2659836.0,63185.0,46.01008,8.96004,Europe/Zurich,1,name_exact:Lugano
5,Lausanne,NaN,NaN,CHE,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,...,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...,CH,2659994.0,139111.0,46.51600,6.63282,Europe/Zurich,1,name_exact:Lausanne
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,Indore,NaN,NaN,IND,India,"Indore, India",18.1,4.0,11.6,20.5,...,60.2,https://www.numbeo.com/cost-of-living/in/Indor...,IN,1269743.0,1994397.0,22.71792,75.83330,Asia/Kolkata,1,name_exact:Indore
621,Guwahati,NaN,NaN,IND,India,"Guwahati, India",17.7,3.3,11.0,21.3,...,75.8,https://www.numbeo.com/cost-of-living/in/Guwah...,IN,1271476.0,962334.0,26.18440,91.74580,Asia/Kolkata,1,name_exact:Guwahati
622,Lucknow,NaN,NaN,IND,India,"Lucknow (Lakhnau), India",17.5,3.4,11.0,19.8,...,77.9,https://www.numbeo.com/cost-of-living/in/Luckn...,IN,1264733.0,2472011.0,26.83928,80.92313,Asia/Kolkata,1,name_exact:Lucknow


Проверка. Это покажет, сколько пропусков в каждом geo-поле.

In [8]:
geo_cols = ["geonameid", "population", "latitude", "longitude", "timezone"]

print("Пустые значения по geo-полям:")
display(df_2026_final_full[geo_cols].isna().sum().to_frame("null_count"))

Пустые значения по geo-полям:


,null_count
geonameid,1
population,1
latitude,1
longitude,1
timezone,1


## Теперь проверка на главный инвариант:

#### 1. Есть geonameid, но чего-то из остальных полей не хватает

In [9]:
df_geo_incomplete = df_2026_final_full[
    df_2026_final_full["geonameid"].notna()
    & (
        df_2026_final_full["population"].isna()
        | df_2026_final_full["latitude"].isna()
        | df_2026_final_full["longitude"].isna()
        | df_2026_final_full["timezone"].isna()
    )
].copy()

print("Есть geonameid, но не заполнены остальные geo-поля:", len(df_geo_incomplete))

display(
    df_geo_incomplete[
        [
            "city",
            "country",
            "city_country",
            "geonameid",
            "population",
            "latitude",
            "longitude",
            "timezone",
            "match_source",
        ]
    ]
)


Есть geonameid, но не заполнены остальные geo-поля: 0


,city,country,city_country,geonameid,population,latitude,longitude,timezone,match_source
city_id,,,,,,,,,


#### 2. Нет geonameid, но почему-то что-то из остальных полей заполнено

In [10]:
df_geo_weird = df_2026_final_full[
    df_2026_final_full["geonameid"].isna()
    & (
        df_2026_final_full["population"].notna()
        | df_2026_final_full["latitude"].notna()
        | df_2026_final_full["longitude"].notna()
        | df_2026_final_full["timezone"].notna()
    )
].copy()

print("Нет geonameid, но какие-то geo-поля заполнены:", len(df_geo_weird))

display(
    df_geo_weird[
        [
            "city",
            "country",
            "city_country",
            "geonameid",
            "population",
            "latitude",
            "longitude",
            "timezone",
            "match_source",
        ]
    ]
)

Нет geonameid, но какие-то geo-поля заполнены: 0


,city,country,city_country,geonameid,population,latitude,longitude,timezone,match_source
city_id,,,,,,,,,


#### 3. Полностью ненайденные строки

In [11]:
df_geo_not_found = df_2026_final_full[
    df_2026_final_full["geonameid"].isna()
].copy()

print("Полностью не сматчились:", len(df_geo_not_found))

display(
    df_geo_not_found[
        [
            "city",
            "state_code",
            "state_name",
            "country",
            "country_code",
            "country_code2",
            "city_country",
            "match_source",
        ]
    ]
)

Полностью не сматчились: 1


,city,state_code,state_name,country,country_code,country_code2,city_country,match_source
city_id,,,,,,,,
586,Bali,NaN,NaN,Indonesia,IDN,ID,"Bali, Indonesia",not_found


#### 4. Быстрая сводка по качеству

In [12]:
total_rows = len(df_2026_final_full)
matched_rows = df_2026_final_full["geonameid"].notna().sum()
not_matched_rows = df_2026_final_full["geonameid"].isna().sum()
incomplete_rows = len(df_geo_incomplete)
weird_rows = len(df_geo_weird)

summary = pd.DataFrame(
    {
        "metric": [
            "total_rows",
            "matched_rows",
            "not_matched_rows",
            "incomplete_rows",
            "weird_rows",
        ],
        "value": [
            total_rows,
            matched_rows,
            not_matched_rows,
            incomplete_rows,
            weird_rows,
        ],
    }
)

display(summary)

,metric,value
0,total_rows,527
1,matched_rows,526
2,not_matched_rows,1
3,incomplete_rows,0
4,weird_rows,0


#### 5. Самая строгая проверка

Если хочешь проверить именно правило
“либо заполнены все 5 geo-полей, либо не заполнено ничего”,

In [13]:
df_2026_final_full[
    df_2026_final_full["city_country"] == "Bali, Indonesia"
]

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,...,pollution,link,country_code2,geonameid,population,latitude,longitude,timezone,match_count,match_source
city_id,,,,,,,,,,,,,,,,,,,,,
586,Bali,NaN,NaN,IDN,Indonesia,"Bali, Indonesia",35.7,30.9,33.5,47.4,...,79.0,https://www.numbeo.com/cost-of-living/in/Bali?...,ID,NaN,NaN,NaN,NaN,NaN,0,not_found


In [14]:
geo_cols = ["geonameid", "population", "latitude", "longitude", "timezone"]

geo_null_counts_per_row = df_2026_final_full[geo_cols].isna().sum(axis=1)

df_geo_mixed = df_2026_final_full[
    (geo_null_counts_per_row > 0) & (geo_null_counts_per_row < len(geo_cols))
].copy()

print("Строки с частично заполненными geo-полями:", len(df_geo_mixed))

display(
    df_geo_mixed[
        [
            "city",
            "country",
            "city_country",
            "geonameid",
            "population",
            "latitude",
            "longitude",
            "timezone",
            "match_source",
        ]
    ]
)

Строки с частично заполненными geo-полями: 0


,city,country,city_country,geonameid,population,latitude,longitude,timezone,match_source
city_id,,,,,,,,,


### Я бы запустил именно такой минимальный блок

In [15]:
geo_cols = ["geonameid", "population", "latitude", "longitude", "timezone"]

print("Null count по geo-полям:")
display(df_2026_final_full[geo_cols].isna().sum().to_frame("null_count"))

geo_null_counts_per_row = df_2026_final_full[geo_cols].isna().sum(axis=1)

df_geo_mixed = df_2026_final_full[
    (geo_null_counts_per_row > 0) & (geo_null_counts_per_row < len(geo_cols))
].copy()

df_geo_not_found = df_2026_final_full[
    df_2026_final_full["geonameid"].isna()
].copy()

print("Частично заполненные geo-строки:", len(df_geo_mixed))
print("Полностью не заполненные geo-строки:", len(df_geo_not_found))

display(
    df_geo_mixed[
        [
            "city",
            "country",
            "city_country",
            "geonameid",
            "population",
            "latitude",
            "longitude",
            "timezone",
            "match_source",
        ]
    ]
)

display(df_geo_not_found[["city", "country", "city_country", "match_source"]])


Null count по geo-полям:


,null_count
geonameid,1
population,1
latitude,1
longitude,1
timezone,1


Частично заполненные geo-строки: 0
Полностью не заполненные geo-строки: 1


,city,country,city_country,geonameid,population,latitude,longitude,timezone,match_source
city_id,,,,,,,,,


,city,country,city_country,match_source
city_id,,,,
586,Bali,Indonesia,"Bali, Indonesia",not_found


### Вывод: нет Бали.
Значит добавлю вручную.

In [16]:
manual_overrides = {
    "Bali, Indonesia": {
        "geonameid": 1650534,
        "population": 4225384,
        "latitude": -8.33333,
        "longitude": 115,
        "timezone": "Asia/Makassar",
        "match_count": 1,
        "match_source": "manual_override",
    }
}

In [17]:
for city_country, values in manual_overrides.items():
    mask = df_2026_final_full["city_country"] == city_country
    for col, val in values.items():
        df_2026_final_full.loc[mask, col] = val

Проверяю. Данные по Бали добавились.

In [18]:
df_2026_final_full[
    df_2026_final_full["city_country"] == "Bali, Indonesia"
]

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,...,pollution,link,country_code2,geonameid,population,latitude,longitude,timezone,match_count,match_source
city_id,,,,,,,,,,,,,,,,,,,,,
586,Bali,NaN,NaN,IDN,Indonesia,"Bali, Indonesia",35.7,30.9,33.5,47.4,...,79.0,https://www.numbeo.com/cost-of-living/in/Bali?...,ID,1650534.0,4225384.0,-8.33333,115.0,Asia/Makassar,1,manual_override


Запись таблицы в файл

In [ ]:
df_2026_final_full.to_pickle("./data/geonameid.pkl")

In [21]:
# Проверка
df_read = pd.read_pickle("./data/geonameid.pkl")
df_read

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,...,pollution,link,country_code2,geonameid,population,latitude,longitude,timezone,match_count,match_source
city_id,,,,,,,,,,,,,,,,,,,,,
3,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,...,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CH,2657896.0,415367.0,47.36667,8.55000,Europe/Zurich,1,alt_exact:Zurich
8,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,...,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CH,2660646.0,201741.0,46.20222,6.14569,Europe/Zurich,1,alt_exact:Geneva
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,...,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CH,2661604.0,177595.0,47.55839,7.57327,Europe/Zurich,1,name_exact:Basel
535,Lugano,NaN,NaN,CHE,Switzerland,"Lugano, Switzerland",113.8,44.2,81.6,109.9,...,NaN,https://www.numbeo.com/cost-of-living/in/Lugan...,CH,2659836.0,63185.0,46.01008,8.96004,Europe/Zurich,1,name_exact:Lugano
5,Lausanne,NaN,NaN,CHE,Switzerland,"Lausanne, Switzerland",113.0,52.8,85.2,111.4,...,NaN,https://www.numbeo.com/cost-of-living/in/Lausa...,CH,2659994.0,139111.0,46.51600,6.63282,Europe/Zurich,1,name_exact:Lausanne
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,Indore,NaN,NaN,IND,India,"Indore, India",18.1,4.0,11.6,20.5,...,60.2,https://www.numbeo.com/cost-of-living/in/Indor...,IN,1269743.0,1994397.0,22.71792,75.83330,Asia/Kolkata,1,name_exact:Indore
621,Guwahati,NaN,NaN,IND,India,"Guwahati, India",17.7,3.3,11.0,21.3,...,75.8,https://www.numbeo.com/cost-of-living/in/Guwah...,IN,1271476.0,962334.0,26.18440,91.74580,Asia/Kolkata,1,name_exact:Guwahati
622,Lucknow,NaN,NaN,IND,India,"Lucknow (Lakhnau), India",17.5,3.4,11.0,19.8,...,77.9,https://www.numbeo.com/cost-of-living/in/Luckn...,IN,1264733.0,2472011.0,26.83928,80.92313,Asia/Kolkata,1,name_exact:Lucknow
